# ZooMS Clustering Analysis & Results Interpretation
This notebook visualizes and interprets the results of multiple unsupervised clustering methods on ZooMS binary peak lists.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set project paths
RESULTS_DIR = Path("results")

# Load data
metrics_df = pd.read_csv(RESULTS_DIR / "evaluation_metrics.csv", index_col=0)
meta_df = pd.read_csv(RESULTS_DIR / "sample_metadata_phase1.csv").set_index("Sample Name")
cluster_df = pd.read_csv(RESULTS_DIR / "clustering_results.csv", index_col=0)

print("Results loaded successfully.")

## 1. Comparative Performance Metrics
We compare multiple algorithms: K-Means, Hierarchical (Ward/Euclidean), Hierarchical (Average/Jaccard), Spectral, and HDBSCAN.

In [ ]:
display(metrics_df.sort_values("V-Measure", ascending=False))

metrics_df[["ARI", "V-Measure"]].plot(kind="bar", figsize=(12, 6))
plt.title("Comparison of Clustering Methods")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2. Best Performer Analysis: Spectral Clustering
Spectral clustering performed best. Let's see its contingency table to understand where it succeeds or fails.

In [ ]:
spec_table = pd.read_csv(RESULTS_DIR / "contingency_Spectral_Cluster.csv", index_col=0)

plt.figure(figsize=(14, 8))
sns.heatmap(spec_table, annot=True, cmap="YlGnBu", fmt="d")
plt.title("Spectral Clustering: Cluster vs. Taxonomic Family")
plt.show()

## 3. Visualization: Truth vs. Prediction (t-SNE)
Using t-SNE to compare the actual taxonomic groupings against the Spectral clusters.

In [ ]:
from src.dim_reduction import DimensionalityReducer
X_binary = pd.read_csv(RESULTS_DIR / "feature_matrix_binary.csv", index_col=0)
reducer = DimensionalityReducer()

tsne_df = reducer.run_tsne(X_binary)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.scatterplot(x=tsne_df.iloc[:,0], y=tsne_df.iloc[:,1], hue=meta_df["Correct ID"], ax=axes[0], palette="tab10", alpha=0.8)
axes[0].set_title("t-SNE: Colored by Ground Truth Family")
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

sns.scatterplot(x=tsne_df.iloc[:,0], y=tsne_df.iloc[:,1], hue=cluster_df["Spectral_Cluster"].astype(str), ax=axes[1], palette="tab20", alpha=0.8)
axes[1].set_title("t-SNE: Colored by Spectral Cluster ID")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

## 4. Key Takeaways
1. **Spectral Clustering Excellence**: The non-linear nature of Spectral clustering captured taxonomic structures better than centroid-based (K-Means) or density-based (HDBSCAN) methods.
2. **Binary Limitations**: The poor performance of Jaccard-based HDBSCAN suggests that the binary matrix might benefit from higher-quality pre-processing or that family separation depends on intensity levels more than expected.
3. **Future Work**: Evaluate the **TIC-Log** matrix using Spectral clustering to see if intensity information pushes the V-Measure above 0.7.